In [1]:
from datetime import datetime
# TENSORFLOW AND KERAS THINGS
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras import layers
import numpy as np
import pandas as pd
import tensorflow as tf
from dqn_utils.utils import create_q_model, create_q_model_second_model

# ENV THINGS
from adabyron_environment.mercadoSecundario_v2 import MercadoSecundario
from adabyron_environment.agents.storageBattery_mercado_secundario import StorageBatteryMS
import matplotlib.pyplot as plt  

2025-06-19 13:38:07.286922: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750333087.297998  264569 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750333087.301355  264569 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-19 13:38:07.313148: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
#################################################
######### ENVIRONMENT INITIALIZATION ############
#################################################
bolsa_meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

bandaSecundaria_df = pd.read_csv('test_data/bandaSecundaria.csv', sep=';')
regulacionSubir_df = pd.read_csv('test_data/regulacionSubir.csv', sep=';')
regulacionBajar_df = pd.read_csv('test_data/regulacionBajar.csv', sep=';')
porcentajeSubir_df = pd.read_csv('test_data/porcentajeSubir.csv', sep=';')
porcentajeBajar_df = pd.read_csv('test_data/porcentajeBajar.csv', sep=';')

In [3]:
#################################################
######### DQN MODEL INITIALIZATION ##############
#################################################

model = create_q_model_second_model(3)

model.load_weights("models/gedera_model.h5")


/home/<user>/Python_Envs/gedera/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1750333088.617776  264569 gpu_device.cc:2344] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
def transform_month_history(month_history):
    month = []
    for day in month_history:
        actual_day = day
        cumulative_price = []
        cumulative_consumption = []
        soc = []
        action = []
        consumption = []
        
        while actual_day.father is not None:
            cumulative_price.append(float(actual_day.cumulative_price))
            cumulative_consumption.append(float(actual_day.cumulative_consumption))
            soc.append(float(actual_day.agents_info['B1']['soc']))
            if actual_day.actions['B1'] == 0:
                action.append("skip")
            elif actual_day.actions['B1'] == 1:
                action.append("charge")
            elif actual_day.actions['B1'] == 2:
                action.append("discharge")
            consumption.append(float(actual_day.consumption))
            
            actual_day = actual_day.father
            
        day_dict = {
            "benefit": float(day.cumulative_price)
        }
            
        cumulative_price.reverse()
        cumulative_consumption.reverse()
        soc.reverse()
        action.reverse()
        consumption.reverse()
        
        day_dict['cumulative_price'] = cumulative_price
        day_dict['cumulative_consumption'] = cumulative_consumption
        day_dict['soc'] = soc
        day_dict['action'] = action
        day_dict['consumption'] = consumption
        
        month.append(day_dict)
    return month
        
    

In [5]:
#################################################
######### DQN TESTING  ##########################
#################################################
original_soc = 23

resultados = {}

for i, month in enumerate(bolsa_meses):
    
    bandaSecundaria = bandaSecundaria_df[month]
    regulacionSubir = regulacionSubir_df[month]
    regulacionBajar = regulacionBajar_df[month]
    porcentajeSubir = porcentajeSubir_df[month]
    porcentajeBajar = porcentajeBajar_df[month]
    
    energy_fixed_input = np.array([bandaSecundaria, regulacionSubir, porcentajeSubir, regulacionBajar, porcentajeBajar]).flatten()
    
    battery_capacity = 36
    
    month_history = []
    
    
    for day in range(0, 30):    
        
        test_battery = StorageBatteryMS("B1",capacity= battery_capacity, original_soc = original_soc)

        
        mercadoSecundario = MercadoSecundario(preservedEnergy=0.50, 
                                                priceCurve= bandaSecundaria, 
                                                highCurve= regulacionSubir, 
                                                lowCurve= regulacionBajar,
                                                highPercentage= porcentajeSubir,
                                                lowPercentage= porcentajeBajar,
                                                battery= test_battery)


        proto_state = mercadoSecundario.reset()
        state = [proto_state.day_step, proto_state.cumulative_consumption, proto_state.cumulative_price, proto_state.agents_info['B1']['soc'], battery_capacity, proto_state.agents_info['B1']['chargeHistory'], proto_state.agents_info['B1']['dischargeHistory']] + energy_fixed_input.tolist()
        
        while True:
            state_tensor = tf.convert_to_tensor([state])
            action_probs = model(state_tensor, training=False)
            action = tf.argmax(action_probs[0]).numpy()
            # Apply the sampled action in our environment
            proto_state_next, reward, done = mercadoSecundario.step_dqn(proto_state, action)
            state_next = [proto_state_next.day_step, proto_state_next.cumulative_consumption, proto_state_next.cumulative_price, proto_state_next.agents_info['B1']['soc'], battery_capacity, proto_state_next.agents_info['B1']['chargeHistory'], proto_state_next.agents_info['B1']['dischargeHistory']] + energy_fixed_input.tolist()
            state       = state_next
            proto_state = proto_state_next 
            if done: 
                month_history.append(proto_state_next)
                original_soc = proto_state_next.agents_info['B1']['soc']
                break
    

    fixed_month = transform_month_history(month_history)
        
    resultados[month] = fixed_month  

with open("resultados.json", "w") as f:
    import json
    
    json.dump(resultados, f, indent=4)

In [6]:
# soc = []
# cumulative_price = []
# for idx, state in enumerate(history):
#     if idx % 25 == 0:
#         continue
#     soc.append(state.agents_info["B1"]["soc"])
#     cumulative_price.append(state.cumulative_price)

In [7]:
# print(cumulative_price[75])
# plt.figure(figsize=(20,5))
# plt.plot(cumulative_price)
# plt.show()

In [8]:
# plt.figure(figsize=(20,5))
# plt.plot(soc)
# plt.show()

In [9]:
# tiempos_2 = [i.total_seconds() for i in tiempos]
# tiempos_2.append(tiempos_2[6])
# tiempos_2.append(tiempos_2[9])
# tiempos_2.append(tiempos_2[5])
# tiempos_2[0] = tiempos_2[4]
# plt.plot(tiempos_2)
# plt.xlabel('Pasos')
# plt.ylabel('Tiempo (s)')
# plt.title('Tiempo de ejecución en función de los pasos')
# plt.savefig("figurita.png")